# AlleleMatrix — brapi-py

Interactive notebook for exploring the BrAPI **AlleleMatrix** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Uncomment to install / upgrade brapi-py
# %pip install brapi-py
import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient

# ── Edit these ────────────────────────────────────────────────────────────
BASE_URL       = 'https://brapi.example.com'
TOKEN_ENDPOINT = 'https://auth.example.com/token'
CLIENT_ID      = 'my-client'
CLIENT_SECRET  = 'my-secret'
# ─────────────────────────────────────────────────────────────────────────

client = BrapiClient(
    base_url=BASE_URL,
    token_endpoint=TOKEN_ENDPOINT,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)
print('Client ready →', client)

## 2 — Search  (`POST /search/allelematrix`)

Full-featured search. Handles both synchronous (HTTP 200) and asynchronous (HTTP 202 + polling) responses.

In [ ]:
# Basic search — filtered by preview()
df = (
    client.allele_matrix
    .preview(true)
    .search()
    .to_df()
)
print(f'{len(df)} records returned')
df.head()

In [ ]:
# Immutable builder — fork the same base query safely
base = (
    client.allele_matrix
    .preview(true)
)

q1_df = base.data_matrix_names(['Genotype']).search().to_df()
q2_df = base.data_matrix_abbreviations(['GT']).search().to_df()

print('q1:', len(q1_df), '  q2:', len(q2_df))

In [ ]:
# Bulk filter — apply multiple criteria in one call
df = (
    client.allele_matrix
    .filter(
        preview=true,
        data_matrix_names=['Genotype'],
        data_matrix_abbreviations=['GT'],  # add more filters above
    )
    .search()
    .to_df()
)
df.head(10)

## 5 — Pipe transforms

`.pipe()` applies a pure function lazily — only one HTTP call is made regardless of how many transform stages are chained.

In [ ]:
def filter_non_null_ids(items):
    """Keep only records with a non-empty primary key."""
    return [r for r in items if r.]

def add_label(items):
    """Attach a short display label to each record."""
    for r in items:
        r.label = f'AlleleMatrix {r.}'
    return items

df = (
    client.allele_matrix
    .preview(true)
    .search()
    .pipe(filter_non_null_ids)
    .pipe(add_label)
    .to_df()
)
df.head(10)

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.allele_matrix
    .preview(true)
    .search()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'expandHomozygotes'
if 'expandHomozygotes' in df.columns:
    df['expandHomozygotes'].value_counts().head(10)
else:
    print('Column not present in this result set')

In [ ]:
import json, pandas as pd

# Explode the 'callSets' JSON column into separate rows
col = 'callSets'
if col in df.columns:
    id_col = ''
    exploded = (
        df.filter(items=[id_col, col])
        .dropna(subset=[col])
        .assign(**{col: lambda d: d[col].apply(json.loads)})
        .explode(col)
    )
    exploded.head(10)
else:
    print(f'Column {col!r} not present in this result set')

## 7 — Error handling

In [ ]:
import httpx

# Async search poll timeout
try:
    client.allele_matrix.search(max_poll_attempts=1).to_list()
except TimeoutError as e:
    print('TimeoutError:', e)
print('Error handling examples above — edit as needed')

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')